In [12]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import time

# --- Safe request with retries ---
def safe_request(url, params=None, retries=5):
    delay = 1
    for i in range(retries):
        try:
            r = requests.get(url, params=params, timeout=10)
            r.raise_for_status()
            return r
        except (requests.ConnectionError, requests.Timeout):
            print(f"Connection failed (attempt {i+1}/{retries}) for {url}, retrying in {delay}s...")
            time.sleep(delay)
            delay *= 2
        except requests.HTTPError as e:
            # HTTP errors like 404, 410 → snapshot missing → not a network failure
            return e.response
    print(f"Failed to fetch {url} after {retries} attempts (network issues)")
    return None  # network failure

# --- Get latest snapshot for a given date ---
def get_latest_snapshot(target_date):
    cdx_url = "https://web.archive.org/cdx/search/cdx"
    params = {
        "url": "https://www.marketwatch.com/",
        "from": target_date.strftime("%Y%m%d"),
        "to": target_date.strftime("%Y%m%d"),
        "fl": "timestamp,original",
        "filter": "statuscode:200",
        "collapse": "timestamp:8"
    }
    r = safe_request(cdx_url, params=params)
    
    if r is None:
        return "NETWORK_FAILURE"  # retryable network failure
    
    if r.status_code != 200 or not r.text.strip():
        return None  # snapshot missing
    
    lines = r.text.strip().split("\n")
    if not lines or len(lines[0].split()) == 0:
        return None  # snapshot missing
    
    ts = lines[0].split(" ")[0]
    return f"https://web.archive.org/web/{ts}/https://www.marketwatch.com/"

# --- Extract headlines ---
def get_headlines_flexcard(archive_html):
    soup = BeautifulSoup(archive_html, "html.parser")
    headlines = []
    for a in soup.find_all("a", {"data-testid": "flexcard-headline"}):
        text = a.get_text(strip=True)
        if text:
            headlines.append(text)
    return headlines

# --- Initialize storage ---
all_headlines = {}
failed_days = []

# --- Scrape past 30 days ---
today = datetime.utcnow().date()
start_date = today - timedelta(days=30)

for n in range(5):
    target_date = start_date + timedelta(days=n)
    archive_url = get_latest_snapshot(target_date)
    
    if archive_url and archive_url != "NETWORK_FAILURE":
        r = safe_request(archive_url)
        if r and r.status_code == 200:
            headlines = get_headlines_flexcard(r.text)
            all_headlines[target_date] = headlines
            print(f"{target_date} - Found {len(headlines)} headlines")
        else:
            failed_days.append(target_date)
            print(f"{target_date} - Network failure during scraping")
    
    elif archive_url == "NETWORK_FAILURE":
        failed_days.append(target_date)
        print(f"{target_date} - Network failure fetching snapshot")
    
    # else snapshot missing → do nothing
    time.sleep(2)


/var/folders/cj/18qyskt93d727jykntbfmn8h0000gn/T/ipykernel_52249/176074100.py:65: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow().date()


Connection failed (attempt 1/5) for https://web.archive.org/web/20250930012024/https://www.marketwatch.com/, retrying in 1s...
Connection failed (attempt 2/5) for https://web.archive.org/web/20250930012024/https://www.marketwatch.com/, retrying in 2s...
Connection failed (attempt 3/5) for https://web.archive.org/web/20250930012024/https://www.marketwatch.com/, retrying in 4s...
Connection failed (attempt 4/5) for https://web.archive.org/web/20250930012024/https://www.marketwatch.com/, retrying in 8s...
Connection failed (attempt 5/5) for https://web.archive.org/web/20250930012024/https://www.marketwatch.com/, retrying in 16s...
Failed to fetch https://web.archive.org/web/20250930012024/https://www.marketwatch.com/ after 5 attempts (network issues)
2025-09-30 - Network failure during scraping
Connection failed (attempt 1/5) for https://web.archive.org/cdx/search/cdx, retrying in 1s...
2025-10-01 - Found 51 headlines
2025-10-02 - Found 51 headlines
2025-10-03 - Found 51 headlines


In [23]:
for key in all_headlines.keys():
  a = key

len(all_headlines[a])

51

In [ ]:
import time

def retry_failed_days():
    """Retry all network failures in failed_days list."""
    global failed_days, all_headlines
    still_failed = []

    for target_date in failed_days:
        archive_url = get_latest_snapshot(target_date)
        if archive_url:
            r = safe_request(archive_url)
            if r and r.status_code == 200:
                headlines = get_headlines_flexcard(r.text)
                if headlines:
                    all_headlines[target_date] = headlines
                    print(f"{target_date} - Successfully fetched ({len(headlines)} headlines)")
                else:
                    still_failed.append(target_date)
            else:
                still_failed.append(target_date)
        else:
            still_failed.append(target_date)
        time.sleep(2)  # polite pause

    failed_days = still_failed
    if failed_days:
        print(f"Remaining network failures: {len(failed_days)}")
    else:
        print("All network failures have been successfully retried.")


In [21]:
retry_failed_days()

2025-09-30 - Successfully fetched (51 headlines)
All network failures have been successfully retried.


In [22]:
len(all_headlines.keys())

4